# CAMB & CosmoPower: exercises (+ a `cloelib` example)

This is the **exercise** version of `CAMB_CosmoPower_cloelib_tutorial_solutions.ipynb`. Wherever
you see a `# TODO` and a line ending in `...`, replace the `...` with working code. Everything else
(installs, imports, a compatibility fix, and all plotting) is provided so you can focus on the
actual API calls.

Each TODO links to the exact page you need:

* **CAMB** — [main docs](https://camb.readthedocs.io) and the official
  [CAMB Demo notebook](https://github.com/cmbant/CAMB/blob/master/docs/CAMBdemo.ipynb) (the
  canonical worked example the CAMB authors themselves publish).
* **CosmoPower** — [docs site](https://alessiospuriomancini.github.io/cosmopower/), in particular
  the [Getting Started with `cosmopower_NN`](https://alessiospuriomancini.github.io/cosmopower/tutorial/getting_started/getting_started_with_cosmopower_NN/getting_started_with_cosmopower_NN/)
  tutorial and the [`cosmopower_NN` API reference](https://alessiospuriomancini.github.io/cosmopower/API/cosmopower_NN-reference/).
* **`cloelib`** — there's no standalone docs site yet; the source itself and the
  [`cloe-org/playground`](https://github.com/cloe-org/playground) repo *are* the documentation.
  Specifically: [`tutorials/cosmology/cosmology.ipynb`](https://github.com/cloe-org/playground/blob/main/tutorials/cosmology/cosmology.ipynb)
  demonstrates exactly the classes you need here, and the
  [`cloelib/cosmology/`](https://github.com/cloe-org/cloelib/tree/main/cloelib/cosmology) source
  directory shows every backend module and class name.

If you get stuck, check the solutions notebook — but try the docs first, that's the point of the
exercise.

> **Colab tip:** the install cell pins an older TensorFlow version. If Colab prompts you to
> **restart the runtime** afterwards, do so once, then re-run from the top.


## 0. Setup (Colab) — provided, just run it

In [ ]:
# Core theory codes
!pip install -q camb cosmopower cosmopower-jax

# cloelib isn't on PyPI yet, so we install it straight from its GitHub repo.
# The [camb,cosmopower-jax] extras pull in the two backends we use below.
!pip install -q "cloelib[camb,cosmopower-jax] @ git+https://github.com/cloe-org/cloelib.git"

# CosmoPower ships its *code* on PyPI, but the pre-trained model *weights* only
# live in its GitHub repo -> shallow-clone just that.
!git clone -q --depth 1 https://github.com/alessiospuriomancini/cosmopower.git cosmopower_repo

print("Setup done.")


## 1. CAMB

We'll define one fiducial flat-$\Lambda$CDM cosmology and reuse it in every section below, so the
three engines (CAMB, CosmoPower, cloelib) can be compared on equal footing.

**🎯 TODO 1** — Build a CAMB parameter object for the cosmology below using `camb.set_params`.
Docs: [`camb.set_params`](https://camb.readthedocs.io/en/latest/camb.html#camb.camb.set_params);
see also the "Set cosmology" cell near the top of the
[CAMB Demo notebook](https://github.com/cmbant/CAMB/blob/master/docs/CAMBdemo.ipynb).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import camb

# Fiducial cosmology (Planck-like), reused throughout the notebook
H0 = 67.7
h = H0 / 100.0
ombh2 = 0.022
omch2 = 0.12
ns = 0.96
As = 2e-9
tau = 0.055
mnu = 0.0

# TODO 1: create the CAMB parameter object from H0, ombh2, omch2, mnu, tau, As, ns above.
pars = ...

print(pars)


### 1a. CMB temperature power spectrum

**🎯 TODO 2** — Extend `pars` to compute lensed spectra up to $\ell_\mathrm{max}=2500$, run CAMB,
and pull out the total $C_\ell$ array. Docs:
[`set_for_lmax`](https://camb.readthedocs.io/en/latest/model.html#camb.model.CAMBparams.set_for_lmax),
[`camb.get_results`](https://camb.readthedocs.io/en/latest/camb.html#camb.camb.get_results),
[`get_cmb_power_spectra`](https://camb.readthedocs.io/en/latest/results.html#camb.results.CAMBdata.get_cmb_power_spectra)
— and the "CMB power spectra" section of the
[CAMB Demo notebook](https://github.com/cmbant/CAMB/blob/master/docs/CAMBdemo.ipynb).


In [ ]:
# TODO 2a: set lmax=2500 with lens_potential_accuracy=1
...

# TODO 2b: run CAMB
results = ...

# TODO 2c: get the CMB power spectra in muK units, and pull out the 'total' entry
# (columns are TT, EE, BB, TE)
totCL = ...

ell = np.arange(totCL.shape[0])
plt.figure(figsize=(7, 4))
plt.plot(ell[2:], totCL[2:, 0])
plt.xlabel(r'Multipole $\ell$')
plt.ylabel(r'$\ell(\ell+1)C_\ell^{TT}/2\pi\ \ [\mu K^2]$')
plt.title('CAMB: lensed CMB temperature power spectrum')
plt.show()


### 1b. Linear & non-linear matter power spectrum

**🎯 TODO 3** — For each of `z_list = [0.0, 0.5, 1.0, 2.0]`, compute the **linear** matter power
spectrum, then switch to **non-linear** (HMcode / `mead2016`) and compute it again. Docs:
[`set_matter_power`](https://camb.readthedocs.io/en/latest/model.html#camb.model.CAMBparams.set_matter_power),
[`NonLinear` options](https://camb.readthedocs.io/en/latest/model.html#camb.model.CAMBparams.NonLinear),
[`get_matter_power_spectrum`](https://camb.readthedocs.io/en/latest/results.html#camb.results.CAMBdata.get_matter_power_spectrum)
— and the "Matter power spectra" section of the
[CAMB Demo notebook](https://github.com/cmbant/CAMB/blob/master/docs/CAMBdemo.ipynb).


In [ ]:
z_list = [0.0, 0.5, 1.0, 2.0]

# TODO 3a: tell `pars` which redshifts and kmax to tabulate P(k) for (kmax=10.0)
...

# TODO 3b: linear P(k) -- set pars.NonLinear to the "none" option, get results, then
# call get_matter_power_spectrum(minkh=1e-4, maxkh=10, npoints=300)
...
kh_lin, z_lin, pk_lin = ...

# TODO 3c: non-linear P(k) -- set pars.NonLinear to "both", set the halofit_version to
# 'mead2016' on pars.NonLinearModel, get results again, and call
# get_matter_power_spectrum(minkh=1e-4, maxkh=10, npoints=300)
...
kh_nl, z_nl, pk_nl = ...

plt.figure(figsize=(7, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(z_list)))
for i, zi in enumerate(z_list):
    plt.loglog(kh_lin, pk_lin[i], color=colors[i], ls='--',
               label=f'linear, z={zi}' if i == 0 else None)
    plt.loglog(kh_nl, pk_nl[i], color=colors[i], ls='-',
               label=f'non-linear, z={zi}' if i == 0 else None)
plt.xlabel('k  [h/Mpc]')
plt.ylabel(r'$P(k)$  [(Mpc/h)$^3$]')
plt.title('CAMB: linear vs. non-linear (Mead2016) matter power spectrum')
plt.legend()
plt.show()


## 2. CosmoPower

CosmoPower's `cosmopower_NN` class loads a small neural network trained to map cosmological
parameters directly to a (log-)power spectrum, bypassing the Boltzmann solve entirely. We'll use
the paper's official pre-trained matter-power-spectrum emulators:

* `PKLIN_NN` — linear $P(k)$
* `PKNLBOOST_NN` — the non-linear/linear boost (trained against HMcode, same as CAMB's `mead2016`
  option above), so `nonlinear = linear * boost`

Note the **unit convention differs from CAMB**: CosmoPower's grid is in $\mathrm{Mpc}^{-1}$ /
$\mathrm{Mpc}^3$ (no factors of $h$), so we convert CAMB's `kh`/`Pk` before comparing (already done
for you below).

> The cell right below is a pre-written compatibility fix, **not** part of the exercise — just run
> it. (It works around CosmoPower's pre-trained files referencing a TensorFlow internal path that
> newer TF versions removed; see
> [cosmopower#22](https://github.com/alessiospuriomancini/cosmopower/issues/22).)


In [ ]:
import sys
import importlib
import importlib.abc
import importlib.util

_OLD_PREFIX = "tensorflow.python.training.tracking"
_NEW_PREFIX = "tensorflow.python.trackable"
_RENAMES = {"tracking": "autotrackable"}


class _TFCompatLoader(importlib.abc.Loader):
    def __init__(self, target_name):
        self.target_name = target_name

    def create_module(self, spec):
        return importlib.import_module(self.target_name)

    def exec_module(self, module):
        pass


class _TFCompatFinder(importlib.abc.MetaPathFinder):
    def find_spec(self, name, path, target=None):
        if name != _OLD_PREFIX and not name.startswith(_OLD_PREFIX + "."):
            return None
        suffix = name[len(_OLD_PREFIX):].lstrip(".")
        parts = suffix.split(".") if suffix else []
        if parts:
            parts[0] = _RENAMES.get(parts[0], parts[0])
        target_name = ".".join([_NEW_PREFIX] + parts) if parts else _NEW_PREFIX
        try:
            importlib.import_module(target_name)
        except ImportError:
            return None
        return importlib.util.spec_from_loader(name, _TFCompatLoader(target_name))


sys.meta_path.insert(0, _TFCompatFinder())
print("TensorFlow trackable-module compatibility shim installed.")


**🎯 TODO 4** — Load both emulators with `cp.cosmopower_NN(restore=True, restore_filename=...)`.
The trained files live under `pk_dir` at `PKLIN_NN` and `PKNLBOOST_NN` (no file extension in the
path). Docs:
[`cosmopower_NN` API reference](https://alessiospuriomancini.github.io/cosmopower/API/cosmopower_NN-reference/)
and the
[Getting Started tutorial](https://alessiospuriomancini.github.io/cosmopower/tutorial/getting_started/getting_started_with_cosmopower_NN/getting_started_with_cosmopower_NN/).

**🎯 TODO 5** — Build the `params_lin` dict the emulator expects. Required keys and their allowed
ranges are listed in the
[PK trained-models README](https://github.com/alessiospuriomancini/cosmopower/blob/main/cosmopower/trained_models/CP_paper/PK/README.md)
on GitHub — note the emulator wants `ln10^{10}A_s` (i.e. $\ln(10^{10}A_s)$), not `As` directly,
and treats `z` as just another input parameter (so every value must be a list of length
`len(z_list)`, matching the other entries).


In [ ]:
import cosmopower as cp

pk_dir = 'cosmopower_repo/cosmopower/trained_models/CP_paper/PK'

# TODO 4: restore both pre-trained emulators
lin_emu = ...
nlboost_emu = ...

k_modes = np.loadtxt(f'{pk_dir}/k_modes.txt')  # Mpc^-1, the fixed grid the NN was trained on

n = len(z_list)
# TODO 5: build the parameter dict the linear emulator expects
# (keys: omega_b, omega_cdm, h, n_s, ln10^{10}A_s, z -- each a list of length n)
params_lin = ...

# Standard HMcode2016 dark-matter-only baryon-feedback calibration (Mead et al. 2016, Table 2) -
# the same defaults CAMB's 'mead2016' halofit_version uses internally. (provided)
params_nlboost = {**params_lin, 'c_min': [3.13] * n, 'eta_0': [0.603] * n}

print(lin_emu, nlboost_emu)


**🎯 TODO 6** — Get predictions from both networks. Both `predictions_np` and
`ten_to_predictions_np` are documented in the
[`cosmopower_NN` API reference](https://alessiospuriomancini.github.io/cosmopower/API/cosmopower_NN-reference/)
— think about which one you need here, given that these networks were trained on **log**-power
spectra (hint: the boost is `nonlinear/linear`, so it combines *additively* in log-space, then you
exponentiate once at the end).


In [ ]:
# TODO 6: predict log P_lin(k) and log-boost, then combine into linear & non-linear P(k)
log_pk_lin = ...
log_boost = ...
pk_cp_lin = ...
pk_cp_nl = ...

print(pk_cp_nl.shape)  # should be (len(z_list), len(k_modes))


### 2a. CosmoPower vs. CAMB, same cosmology — provided, just run it

In [ ]:
plt.figure(figsize=(7, 5))
for i, zi in enumerate(z_list):
    plt.loglog(kh_nl * h, pk_nl[i] / h**3, color=colors[i], ls='-',
               label='CAMB (Mead2016)' if i == 0 else None)
    plt.loglog(k_modes, pk_cp_nl[i], color=colors[i], ls='--',
               label='CosmoPower NN' if i == 0 else None)
plt.xlim(1e-3, 5)
plt.xlabel('k  [1/Mpc]')
plt.ylabel(r'$P(k)$  [Mpc$^3$]')
plt.title('Non-linear matter power spectrum: CAMB vs. CosmoPower emulator')
plt.legend()
plt.show()


### 2b. Why bother: a speed comparison

**🎯 TODO 7** — Time `n_samples` predictions from `lin_emu` in one batched call, versus
`n_samples` individual CAMB runs in a loop. Reuse what you did in TODO 1 and TODO 6.


In [ ]:
import time

n_samples = 50
rng = np.random.default_rng(42)
# draws within the emulator's trained parameter ranges
omb_s = rng.uniform(0.01875, 0.02625, n_samples)
omc_s = rng.uniform(0.05, 0.255, n_samples)
h_s = rng.uniform(0.64, 0.82, n_samples)
ns_s = rng.uniform(0.84, 1.1, n_samples)
lnAs_s = rng.uniform(1.61, 3.91, n_samples)

params_batch = {
    'omega_b': omb_s, 'omega_cdm': omc_s, 'h': h_s, 'n_s': ns_s,
    'ln10^{10}A_s': lnAs_s, 'z': np.zeros(n_samples),
}

t0 = time.time()
# TODO 7a: one batched CosmoPower call over the whole `params_batch`
...
t_cp = time.time() - t0
print(f'CosmoPower: {n_samples} P(k) predictions in {t_cp:.4f} s')

t0 = time.time()
for i in range(n_samples):
    # TODO 7b: build a CAMB param object from omb_s[i]/omc_s[i]/h_s[i]/ns_s[i]/lnAs_s[i],
    # set matter power at z=0 with kmax=2.0, get results, get P(k)
    ...
t_camb = time.time() - t0
print(f'CAMB: {n_samples} P(k) calls in {t_camb:.4f} s')
print(f'Speed-up: {t_camb / t_cp:,.0f}x')


## 3. `cloelib`: one interface, two backends

`cloelib` defines each theory code as a `Background` + `Perturbations` pair implementing a common
API (`hubble_parameter`, `comoving_distance`, `matter_power_spectrum`, `growth_factor`, ...). You'll
build a single `CAMBBackground` for our fiducial cosmology, then compute the matter power spectrum
from it through **two different engines**:

* `CAMBLinearPerturbations` / `CAMBNonLinearPerturbations` — runs CAMB directly (same as Section 1).
* `CosmoPowerJAXLCDMPerturbations.Linear` / `.NonLinear` — runs the
  [`cosmopower-jax`](https://github.com/dpiras/cosmopower-jax) emulator. Its first call downloads
  weights from Zenodo, so it may take a few seconds.

There's no polished docs site for `cloelib` yet — your reference is the
[`tutorials/cosmology/cosmology.ipynb`](https://github.com/cloe-org/playground/blob/main/tutorials/cosmology/cosmology.ipynb)
notebook in `cloe-org/playground`, which builds exactly this kind of `Background`/`Perturbations`
pair (for CAMB, CLASS *and* JAX backends — you only need the CAMB and CosmoPower-JAX pieces here).
If you want to see the class internals directly, browse
[`cloelib/cosmology/camb_cosmology.py`](https://github.com/cloe-org/cloelib/blob/main/cloelib/cosmology/camb_cosmology.py)
and
[`cloelib/cosmology/cosmopower_jax_cosmology.py`](https://github.com/cloe-org/cloelib/blob/main/cloelib/cosmology/cosmopower_jax_cosmology.py)
on GitHub.

**🎯 TODO 8** — Import the four classes you need (`CAMBBackground`, `CAMBLinearPerturbations`,
`CAMBNonLinearPerturbations` from `cloelib.cosmology.camb_cosmology`; `CosmoPowerJAXLCDMPerturbations`
from `cloelib.cosmology.cosmopower_jax_cosmology`).

**🎯 TODO 9** — Construct `background`, matching the `CAMBBackground(...)` call in the playground
`cosmology.ipynb` notebook (arguments: `H0`, `Omega_b0`, `Omega_cdm0`, `Omega_k0`, `As`, `ns`,
`mnu`, `w0`, `wa`, `gamma_MG`, `N_mnu` — flat $\Lambda$CDM means `Omega_k0=0`, `w0=-1`, `wa=0`,
`gamma_MG=0`, `N_mnu=0`, `mnu=0`).


In [ ]:
# TODO 8: imports
...

Omega_b0 = ombh2 / h**2
Omega_cdm0 = omch2 / h**2

# TODO 9: build the shared CAMBBackground
background = ...

print(background)


**🎯 TODO 10** — Compute $P(k, z)$ two ways from the *same* `background`:

1. `CAMBLinearPerturbations(background=..., redshifts=...)` and
   `CAMBNonLinearPerturbations(background=..., redshifts=..., nonlinear_model='mead2016')`, then
   `.matter_power_spectrum(z_arr, ks)` on each.
2. `CosmoPowerJAXLCDMPerturbations.Linear(background=..., redshifts=...)`, then
   `CosmoPowerJAXLCDMPerturbations.NonLinear(background=..., linearperturbations=<the Linear
   instance>, redshifts=..., log10TAGN=7.6)`, then `.matter_power_spectrum(z_arr, ks)` on each.

Check the `cosmology.ipynb` notebook's "CAMB PERTURBATIONS" and "JAX PERTURBATIONS" cells linked
above if the call signatures aren't obvious.


In [ ]:
z_arr = np.linspace(0.0, 2.0, 20)
ks = np.logspace(-3, 1, 200)

# --- Backend 1: cloelib driving CAMB directly ---
camb_linear = ...
camb_nonlinear = ...
pk_camb_lin = ...
pk_camb_nl = ...

# --- Backend 2: cloelib driving the CosmoPower-JAX emulator, off the *same* background ---
cp_linear = ...
cp_nonlinear = ...
pk_cp_lin = ...
pk_cp_nl = ...

print('CAMB backend :', camb_nonlinear)
print('CosmoPower-JAX backend :', cp_nonlinear)


Plotting — provided, just run it once TODO 10 is filled in.

In [ ]:
iz = 0  # z = 0 slice

plt.figure(figsize=(7, 5))
plt.loglog(ks, pk_camb_lin[iz], label='cloelib + CAMB, linear', ls='--')
plt.loglog(ks, pk_camb_nl[iz], label='cloelib + CAMB, non-linear', ls='-')
plt.loglog(ks, pk_cp_lin[iz], label='cloelib + CosmoPower-JAX, linear', ls=':')
plt.loglog(ks, pk_cp_nl[iz], label='cloelib + CosmoPower-JAX, non-linear', ls='-.')
plt.xlabel('k  [1/Mpc]')
plt.ylabel(r'$P(k, z=0)$  [Mpc$^3$]')
plt.title('cloelib: one Background, two theory engines')
plt.legend()
plt.show()


### 3a. Growth rate, both ways

**🎯 TODO 11** — Both backends expose the linear growth rate $f(z)$ through the same method name
(`growth_rate()`, called on the *linear* perturbations instance, no arguments) — again computed
from the one shared `background`. Compare `camb_linear` and `cp_linear`.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(z_arr, ..., label='CAMB', ls='-')       # TODO 11a
plt.plot(z_arr, ..., label='CosmoPower-JAX', ls='--')  # TODO 11b
plt.xlabel('z')
plt.ylabel('f(z)')
plt.title('Growth rate: cloelib + CAMB vs. cloelib + CosmoPower-JAX')
plt.legend()
plt.show()


## References

* CAMB — [Lewis, Challinor & Lasenby 2000](https://arxiv.org/abs/astro-ph/9911177); docs:
  <https://camb.readthedocs.io>; demo notebook:
  <https://github.com/cmbant/CAMB/blob/master/docs/CAMBdemo.ipynb>
* CosmoPower — [Spurio Mancini et al. 2022](https://arxiv.org/abs/2106.03846); docs:
  <https://alessiospuriomancini.github.io/cosmopower/>; code:
  <https://github.com/alessiospuriomancini/cosmopower>; JAX version:
  <https://github.com/dpiras/cosmopower-jax>
* `cloelib` — Euclid Collaboration, *Cosmology Likelihood for Observables in Euclid*. Actively-maintained code:
  <https://github.com/cloe-org/cloelib>; worked examples:
  <https://github.com/cloe-org/playground>; full likelihood/sampler wrapper:
  <https://github.com/cloe-org/cloelike>.

Compare your answers against `CAMB_CosmoPower_cloelib_tutorial_solutions.ipynb`.
